# imports

In [ ]:
import os
import zipfile
import urllib.request
import random
import torchvision
import numpy as np
from PIL import Image
import cv2
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from skimage.feature import local_binary_pattern
from skimage.filters.rank import entropy
from skimage.morphology import disk
import torchvision.transforms as transforms
from dataset import MoCoTextureDataset

In [ ]:
!git pull

# sync github

In [ ]:
repo_name = "SIV-Texture-Anomaly-detection"
git_url = "https://github.com/Marco1743/SIV-Texture-Anomaly-detection.git"
if not os.path.exists(repo_name):
    !git clone {git_url}
%cd {repo_name}
!git pull

c:\Users\marco\Desktop\SIV\SIV-Texture-Anomaly-detection\SIV-Texture-Anomaly-detection


Cloning into 'SIV-Texture-Anomaly-detection'...


Already up to date.


# import dataset

In [ ]:
os.makedirs('data/train/good', exist_ok=True)
os.makedirs('data/test/anomaly', exist_ok=True)

print("Scaricamento Texture reali (DTD Dataset)...")
dtd_train = torchvision.datasets.DTD(root='./data', split='train', download=True)
dtd_test = torchvision.datasets.DTD(root='./data', split='test', download=True)

Download dataset in corso...
Estrazione...


'gdown' is not recognized as an internal or external command,
operable program or batch file.


Dataset pronto!
Cartelle trovate:


'unzip' is not recognized as an internal or external command,
operable program or batch file.
'ls' is not recognized as an internal or external command,
operable program or batch file.


## creazione train e test

In [ ]:
print("creazione training")
count = 0
for img, label in dtd_train:
    img_gray = img.convert('L')
    img_gray.save(f'data/train/good/tex_{count}.png')
    count += 1

print("Generazione anomalie...")
for i in range(50):
    img, _ = dtd_test[i]
    img_arr = np.array(img.convert('L'))
    h, w = img_arr.shape
    
    tipo_anomalia = random.choice([0, 1])
    
    if tipo_anomalia == 0:
        box_h = random.randint(10, 50)
        box_w = random.randint(10, 50)
        y = random.randint(0, h - box_h)
        x = random.randint(0, w - box_w)
        intensita = random.randint(0, 60)
        
        img_arr[y:y+box_h, x:x+box_w] = intensita
        
    else:
        spessore = random.randint(2, 6)
        lunghezza = random.randint(40, 100)
        
        if random.choice([True, False]):
            y = random.randint(0, h - spessore)
            x = random.randint(0, w - lunghezza)
            img_arr[y:y+spessore, x:x+lunghezza] = 255
        else: 
            y = random.randint(0, h - lunghezza)
            x = random.randint(0, w - spessore)
            img_arr[y:y+lunghezza, x:x+spessore] = 255

    Image.fromarray(img_arr).save(f'data/test/anomaly/anom_{i}.png')

print(f"Dataset pronto e processato con augmentations.py! Totale: {count}")

# PIPELINE 1

## Carichiamo l'immagine

In [ ]:
img_path = 'data/test/anomaly/anom_0.png'
img = cv2.imread(img_path, 0)

## Estrazione LBP

In [ ]:
radius = 3
n_points = 8 * radius
lbp_img = local_binary_pattern(img, n_points, radius, method='uniform')

## Entropia Locale sull'LBP

In [ ]:
# Convertiamo l'LBP in un formato digeribile dal filtro di entropia (uint8 0-255)
lbp_uint8 = cv2.normalize(lbp_img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
# Calcoliamo l'entropia (il disordine) in una finestra circolare (raggio 15 pixel)
# Il graffio romperà la regolarità delle strisce, creando un picco di entropia!
entropia_locale = entropy(lbp_uint8, disk(15))

## Segmentazione K-Means sulla mappa di Entropia

In [ ]:
X = entropia_locale.reshape((-1, 1))
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
kmeans.fit(X)
anomaly_mask = kmeans.labels_.reshape(img.shape)
# Trucchetto: Assicuriamoci che l'anomalia sia sempre bianca (1) e lo sfondo nero (0)
# Poiché le anomalie sono piccole, il cluster "anomalo" avrà sempre meno pixel dello sfondo
if np.sum(anomaly_mask) > (anomaly_mask.size / 2):
    anomaly_mask = 1 - anomaly_mask

## Visualizzazione

In [ ]:
plt.figure(figsize=(20, 5))

plt.subplot(1, 4, 1)
plt.title("Immagine con Anomalia", fontsize=14)
plt.imshow(img, cmap='gray')
plt.axis('off')

plt.subplot(1, 4, 2)
plt.title("1. Feature LBP (Cruda)", fontsize=14)
plt.imshow(lbp_img, cmap='viridis')
plt.axis('off')

plt.subplot(1, 4, 3)
plt.title("2. Entropia Locale (Il Segreto!)", fontsize=14)
plt.imshow(entropia_locale, cmap='inferno')
plt.axis('off')

plt.subplot(1, 4, 4)
plt.title("3. Segmentazione K-Means", fontsize=14)
plt.imshow(anomaly_mask, cmap='gray')
plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
print("Inizializzazione del PyTorch Dataset per MoCo...")
train_dataset = MoCoTextureDataset(data_dir='data/train/good', is_train=True)
view_1, view_2 = train_dataset[5]

to_pil = transforms.ToPILImage()
img_1 = to_pil(view_1)
img_2 = to_pil(view_2)

plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.title("Vista 1 (Augmentation Casuale A)", fontsize=14)
plt.imshow(img_1, cmap='gray')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title("Vista 2 (Augmentation Casuale B)", fontsize=14)
plt.imshow(img_2, cmap='gray')
plt.axis('off')

plt.tight_layout()
plt.show()

print(f"Shape del tensore 1: {view_1.shape}")
print(f"Shape del tensore 2: {view_2.shape}")